# Lab A. Runge's Phenomenon and Chebyshev Nodes

**How this lab is graded.** Completion, on a 0-3 scale. The graded parts are the
written derivations in Part 2 and the written answers to the prompts marked
**(written)**. Your code is *not* graded: it is a tool to produce the figures and
numbers you then reason about. If your programming background is thin, you may use
an AI assistant to write the code. You are still responsible for the mathematics,
and the self-check cells exist so you can confirm your code is correct before you
trust its output.

**What to submit.** Run the notebook top to bottom (Runtime > Run all) so every
figure and every self-check output is visible, fill in every cell marked TODO or
(written), and submit the completed `.ipynb`. Do not delete the self-check cells.

In [ ]:
# --- setup ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
mp.mp.dps = 40   # 40-digit reference arithmetic, used in Part 4

## Part 1. Interpolating Runge's function on equispaced nodes

You might expect that using more interpolation nodes always gives a better fit.
Runge's function

$$ f(x) = \frac{1}{1+25x^2} $$

shows this is false. You will build the interpolant on equally spaced nodes and
watch the error as the degree grows.

In [ ]:
def equispaced_nodes(a, b, n):
    """Return the n+1 equally spaced nodes on [a, b], endpoints included.
    Input:  a, b floats with a < b; n an integer >= 0.
    Output: 1D numpy array of length n+1.
    """
    # TODO: your implementation
    raise NotImplementedError

def vandermonde_interpolant(x_nodes, y_nodes):
    """Interpolate the data by solving the Vandermonde system V c = y in the
    monomial basis. Build V yourself and solve the linear system; do not call a
    built-in interpolation routine.
    Input:  x_nodes, y_nodes, 1D arrays of equal length n+1 with distinct x.
    Output: coefficient vector c of length n+1 with p(x) = sum_k c[k]*x**k.
    """
    # TODO: build V with V[i,k] = x_nodes[i]**k, then solve V c = y_nodes
    raise NotImplementedError

In [ ]:
# provided helpers --------------------------------------------------------
def poly_eval(c, x):
    """Evaluate p(x) = sum_k c[k]*x**k by Horner's rule."""
    x = np.asarray(x, float)
    p = np.zeros_like(x)
    for ck in reversed(np.asarray(c, float)):
        p = p * x + ck
    return p

def max_error(f, c, a, b, N=2001):
    """Maximum of |f - p| on a fine grid over [a, b]."""
    xx = np.linspace(a, b, N)
    return np.max(np.abs(f(xx) - poly_eval(c, xx)))

In [ ]:
# === self-check (do not edit): every line must print PASS ===
_x = equispaced_nodes(-1, 1, 5)
assert len(_x) == 6 and np.isclose(_x[0], -1) and np.isclose(_x[-1], 1)
print("PASS  nodes")
_g = lambda x: 2*x**3 - x + 1
_c = vandermonde_interpolant(_x, _g(_x))
assert np.allclose(poly_eval(_c, _x), _g(_x), atol=1e-8)
print("PASS  reproduces the data at the nodes")
assert max_error(_g, _c, -1, 1) < 1e-9          # exact on a cubic
print("PASS  exact on a low-degree polynomial")

In [ ]:
# experiment: overlays and the error curve --------------------------------
def runge(x): return 1.0 / (1.0 + 25.0 * x**2)

xx = np.linspace(-1, 1, 800)
plt.figure(); plt.plot(xx, runge(xx), "k-", lw=2, label="f")
for n in (4, 10, 20):
    xn = equispaced_nodes(-1, 1, n)
    c  = vandermonde_interpolant(xn, runge(xn))
    plt.plot(xx, poly_eval(c, xx), label=f"p_{n}")
plt.ylim(-0.5, 1.5); plt.legend()
plt.title("Equispaced interpolation of Runge's function"); plt.show()

degs, errs = np.arange(0, 26), []
for n in degs:
    xn = equispaced_nodes(-1, 1, n)
    errs.append(max_error(runge, vandermonde_interpolant(xn, runge(xn)), -1, 1))
plt.figure(); plt.semilogy(degs, errs, "r.-")
plt.xlabel("degree n"); plt.ylabel("max error on [-1, 1]")
plt.title("Equispaced error vs degree"); plt.show()

**Q1.1 (written).** Describe what happens to the interpolant and to the maximum
error as the degree grows from 4 to 25, and say where on the interval the error is
worst. From the trend over $n=0,\dots,25$, predict how $p_{100}$ would behave and
state whether equispaced interpolation converges here.

*Your answer:*

## Part 2. Chebyshev nodes (the derivation)

The interpolation error formula from lecture is

$$ f(x) - p_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\,\omega(x), \qquad \omega(x)=\prod_{k=0}^{n}(x-x_k). $$

The factor $f^{(n+1)}/(n+1)!$ is fixed by $f$, but $\omega$ depends entirely on where
the nodes sit. Part 1 suggests the trouble lives near the endpoints, so we want node
placements that keep $\max_x|\omega(x)|$ small. The following construction, due to
Chebyshev, does exactly that.

*Construction (on $[-1,1]$).* Draw the upper unit semicircle over $[-1,1]$. Place
$n+1$ points at angles $\theta_j=\frac{(2j+1)\pi}{2(n+1)}$, $j=0,\dots,n$, equally
spaced around the semicircle, and project them down onto the $x$-axis.

**Q2.1 (written).** Using the construction, show that the projected nodes are

$$ x_j=\cos\frac{(2j+1)\pi}{2(n+1)},\qquad j=0,\dots,n. $$

*Your answer:*

Define the Chebyshev polynomial $T_m(x)=\cos\!\big(m\arccos x\big)$ for $x\in[-1,1]$.

**Q2.2 (written).** Prove that $T_m$ is a polynomial of degree $m$. *Hint.* From the
cosine addition formula, $\cos((m{+}1)\theta)+\cos((m{-}1)\theta)=2\cos\theta\cos(m\theta)$
with $\theta=\arccos x$ gives the recurrence

$$ T_{m+1}(x)=2x\,T_m(x)-T_{m-1}(x),\qquad T_0=1,\ T_1=x. $$

Use induction on $m$.

*Your answer:*

**Q2.3 (written).** Show that the nodes $x_j$ of Q2.1 are exactly the roots of
$T_{n+1}$. *Hint.* $T_{n+1}(x_j)=\cos\!\big((n{+}1)\theta_j\big)$; evaluate at
$\theta_j=\frac{(2j+1)\pi}{2(n+1)}$.

*Your answer:*

The reason these nodes matter is the following minimax property, which we state
and use but do not prove here.

> **Theorem (Chebyshev minimax).** Among all monic polynomials of degree $n+1$ on
> $[-1,1]$, the one with the smallest maximum absolute value is $2^{-n}T_{n+1}$, with
> $\max_{[-1,1]}|2^{-n}T_{n+1}|=2^{-n}$. Consequently, taking the nodes to be the roots
> of $T_{n+1}$ makes the monic node polynomial $\omega=2^{-n}T_{n+1}$, so
> $\max_{[-1,1]}|\omega|=2^{-n}$, the smallest possible.

You will check this value numerically in Part 3.

In [ ]:
def chebyshev_nodes(a, b, n):
    """Return the n+1 Chebyshev nodes on [a, b]. On [-1,1] they are
    cos((2j+1)*pi/(2(n+1))) for j=0..n; map affinely to [a,b].
    Output: 1D numpy array of length n+1 (any order is fine).
    """
    # TODO: build the [-1,1] nodes with the cosine formula, then map to [a,b]
    raise NotImplementedError

In [ ]:
# === self-check (do not edit) ===
_c = chebyshev_nodes(-1, 1, 7)
assert len(_c) == 8
assert np.all(_c > -1 - 1e-12) and np.all(_c < 1 + 1e-12)
# they must be roots of T_8: cos(8*arccos(x)) = 0
assert np.allclose(np.cos(8 * np.arccos(np.clip(_c, -1, 1))), 0, atol=1e-10)
print("PASS  Chebyshev nodes are the roots of T_(n+1)")

## Part 3. Chebyshev interpolation and the node polynomial

Now repeat the Runge experiment with Chebyshev nodes, and inspect $\omega(x)$
directly to see what changed.

In [ ]:
# Chebyshev interpolation of Runge, and both error curves overlaid ---------
xx = np.linspace(-1, 1, 800)
plt.figure(); plt.plot(xx, runge(xx), "k-", lw=2, label="f")
for n in (4, 10, 20):
    xn = chebyshev_nodes(-1, 1, n)
    c  = vandermonde_interpolant(xn, runge(xn))
    plt.plot(xx, poly_eval(c, xx), label=f"q_{n}")
plt.ylim(-0.5, 1.5); plt.legend()
plt.title("Chebyshev interpolation of Runge's function"); plt.show()

degs = np.arange(0, 26)
err_eq, err_ch = [], []
for n in degs:
    xe = equispaced_nodes(-1, 1, n); err_eq.append(max_error(runge, vandermonde_interpolant(xe, runge(xe)), -1, 1))
    xc = chebyshev_nodes(-1, 1, n);  err_ch.append(max_error(runge, vandermonde_interpolant(xc, runge(xc)), -1, 1))
plt.figure()
plt.semilogy(degs, err_eq, "r.-", label="equispaced")
plt.semilogy(degs, err_ch, "b.-", label="Chebyshev")
plt.xlabel("degree n"); plt.ylabel("max error on [-1, 1]"); plt.legend()
plt.title("Equispaced vs Chebyshev error"); plt.show()

In [ ]:
def omega(x, nodes):
    """Evaluate the node polynomial omega(x) = prod_k (x - nodes[k]).
    x may be a scalar or a numpy array; return the same shape.
    """
    # TODO: form the product over the nodes
    raise NotImplementedError

In [ ]:
# === self-check (do not edit) ===
_nd = equispaced_nodes(-1, 1, 6)
assert np.allclose(omega(_nd, _nd), 0.0, atol=1e-12)     # omega vanishes at its nodes
assert np.isclose(omega(0.3, [0.0, 1.0]), 0.3 * (0.3 - 1.0))
print("PASS  omega")

# compare |omega| for the two node sets at n = 20
xx = np.linspace(-1, 1, 4001)
for name, nd in [("equispaced", equispaced_nodes(-1,1,20)), ("Chebyshev", chebyshev_nodes(-1,1,20))]:
    print(f"{name:>11}: max|omega| = {np.max(np.abs(omega(xx, nd))):.3e}")
print(f"{'':>11}  (Chebyshev target 2^-20 = {2.0**-20:.3e})")
plt.figure()
plt.semilogy(xx, np.abs(omega(xx, equispaced_nodes(-1,1,20))) + 1e-30, "r-", label="equispaced")
plt.semilogy(xx, np.abs(omega(xx, chebyshev_nodes(-1,1,20))) + 1e-30, "b-", label="Chebyshev")
plt.xlabel("x"); plt.ylabel("|omega(x)|,  n = 20"); plt.legend()
plt.title("Node polynomial for the two node sets"); plt.show()

**Q3.1 (written).** Contrast the two error curves. What kind of convergence (if
any) does each show as $n$ grows?

*Your answer:*

**Q3.2 (written).** From your $|\omega|$ plot and the printed $\max|\omega|$ values,
which factor in the error formula does the choice of nodes control, and how does that
explain the difference between the two curves? Where is equispaced $\omega$ largest?

*Your answer:*

## Part 4. Two different failures, and why they are not the same

Two true statements about high-degree interpolation are easy to confuse.

1. *Runge's phenomenon:* on equispaced nodes the interpolation error can diverge as
   $n\to\infty$.
2. *Vandermonde ill-conditioning:* the linear system $Vc=y$ you solve to find the
   monomial coefficients is badly conditioned at high degree.

They sound alike, and a quick guess might blame the Part 1 blow-up on "just
floating-point error." The next two experiments let you decide.

In [ ]:
# provided high-precision reference (barycentric interpolation in mpmath) --
def _runge_true_max_error(n, N=401):
    """True max error of the degree-n equispaced interpolant of Runge, computed
    at 40 digits so rounding cannot be the cause of any blow-up we see."""
    xs = [(-1 + mp.mpf(2)*i/n) for i in range(n+1)] if n > 0 else [mp.mpf(0)]
    ys = [1/(1 + 25*xi**2) for xi in xs]
    w = []
    for j in range(len(xs)):
        prod = mp.mpf(1)
        for k in range(len(xs)):
            if k != j: prod *= (xs[j] - xs[k])
        w.append(1/prod)
    worst = mp.mpf(0)
    for i in range(N):
        xq = mp.mpf(-1) + mp.mpf(2)*i/(N-1)
        hit = None; num = den = mp.mpf(0)
        for j in range(len(xs)):
            d = xq - xs[j]
            if d == 0: hit = ys[j]; break
            t = w[j]/d; num += t*ys[j]; den += t
        val = hit if hit is not None else num/den
        e = abs(1/(1 + 25*xq**2) - val)
        if e > worst: worst = e
    return float(worst)

print("Experiment 4a: equispaced Runge, float64 vs 40-digit arithmetic")
print(f"{'n':>3} {'float64 (Vander)':>18} {'40-digit truth':>16}")
for n in [4, 8, 12, 16, 20, 24]:
    xn = equispaced_nodes(-1, 1, n)
    ef = max_error(runge, vandermonde_interpolant(xn, runge(xn)), -1, 1)
    print(f"{n:>3} {ef:18.3e} {_runge_true_max_error(n):16.3e}")

In [ ]:
print("Experiment 4b: Vandermonde conditioning vs interpolation quality")
print(f"{'n':>3} {'cond(V) equisp.':>16} {'exp@Cheb error':>16}")
for n in [4, 8, 12, 16, 20, 24, 28]:
    V = np.vander(equispaced_nodes(-1, 1, n), n + 1, increasing=True)
    xc = chebyshev_nodes(-1, 1, n)
    err = max_error(np.exp, vandermonde_interpolant(xc, np.exp(xc)), -1, 1)
    print(f"{n:>3} {np.linalg.cond(V):16.2e} {err:16.2e}")

**Q4.1 (written).** Use the two experiments to answer: is the equispaced blow-up of
Part 1 caused by rounding error, or by the mathematics of node placement? And does a
large $\mathrm{cond}(V)$ by itself mean the interpolation is inaccurate? Justify each
answer by pointing to specific numbers in 4a and 4b, and state in one sentence how
Runge's phenomenon and Vandermonde ill-conditioning differ.

*Your answer:*

## Where this goes next

Node placement is a design choice, and the node polynomial $\omega$ is the quantity it
controls. The same minimax idea reappears when we choose quadrature nodes, and the
conditioning question from Part 4 returns in full when we study linear systems. Later
homework will assume you have run this lab.